# General-Purpose Fortran-to-Python Transformation Framework

This notebook presents the objectives, architecture, and guiding principles of the **General-Purpose Transformation Framework**. Its primary goal is to translate a large, tightly coupled Fortran codebase into clean, modular Python representations.

The framework focuses on extracting individual Fortran subroutines or functions and converting them into self-contained Python classes. By isolating computational kernels from the original monolithic codebase, the resulting classes become easier to understand, maintain, test, and extend.

Once translated, these Python classes can be further transformed into JIT-compiled modules, enabling high-performance execution, efficient emulation, automatic differentiation, and integration with modern scientific computing ecosystems. This transformation pipeline provides a pathway for modernizing legacy Fortran applications while preserving their original computational behavior.


In [1]:
import ast
import jax
import equinox as eqx 
import jax.numpy as jnp
from jax import jit
import numpy as np

In [ ]:
# This is to ensure float64 is activated, since by default jax uses float32
jax.config.update('jax_enable_x64', True)

In [4]:
%reload_ext autoreload
%autoreload 2
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from fgpt.core.transpiler import F2NP
from fgpt.core.frontend import Processor
from fgpt.core.common import Logger

In [6]:
logger = Logger()
processor = Processor(logger=logger)

In [7]:
f2np_ = F2NP()

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# This is the cost function in Fortran 
cost_function  = """
 SUBROUTINE COST_FUNCTION(u, j)
    IMPLICIT NONE
! Numerical solution at the end time
    REAL, INTENT(IN) :: u
! Cost function value
    REAL, INTENT(OUT) :: j
    INTRINSIC EXP
! The exponential constant, e=2.71828...
    REAL, PARAMETER :: e=EXP(1.0)
    j = (u-e)**2
  END SUBROUTINE COST_FUNCTION
"""

cost_func = processor.parse_fortran_string(cost_function)
print(cost_func)

[INFO] Successfully parsed string!


SUBROUTINE COST_FUNCTION(u, j)
  IMPLICIT NONE
  ! Numerical solution at the end time
  REAL, INTENT(IN) :: u
  ! Cost function value
  REAL, INTENT(OUT) :: j
  INTRINSIC :: EXP
  ! The exponential constant, e=2.71828...
  REAL, PARAMETER :: e = EXP(1.0)
  j = (u - e) ** 2
END SUBROUTINE COST_FUNCTION


In [9]:
_,_,cost_func_python = f2np_.recursive_ast(cost_func)

As you can see, **F2NP** is responsible solely for the translation of code from **Fortran** to **Python**.

After this translation step, the **Transformer** class takes over. Its responsibilities include:

- Ensuring the correctness of inputs and outputs.
- Applying any necessary corrections to the translated code.
- Performing additional transformations and validations as required.

In [10]:
print(ast.unparse(ast.fix_missing_locations(cost_func_python[0])))

def COST_FUNCTION(u, j):
    e = np.float64(np.exp(1.0))
    j = (u - e) ** 2


## Phase 1: Fortran-to-Python Transformation

The first phase of the transformation process consists of converting **Fortran** code into **Python** through an intermediate representation based on Python's **Abstract Syntax Trees (ASTs)**. This representation is implemented in the form of a class that enables structured code translation and manipulation.

The function defined here is taken from the **Differentiable Programming Summer School 2025** materials:

(https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2025/blob/main/session1/notebook.ipynb)

This function is used as a minimal example to illustrate the purpose and workflow of the transformation process.

In [12]:
class Cost:
    # No need for __init__ function since there's no attributes. 
    
    # Cost funciton : (u - exp(1))^2
    def COST_FUNCTION(self, u):
        e = np.exp(1.0)
        j = (u - e) ** 2
        return j
    
    def theta_step(self, u, theta, dt):
        return u * (1 + dt * (1 - theta)) / (1 - dt * theta)
    
    def theta_method(self, theta, u):
        end_time = 1.0
        dt = 0.1 # timestep
        t = 0.0
        u_ = 1.0

        while(t < end_time - 1e-05):
            u = self.theta_step(u_, theta, dt) # Theta-method time-stepping update (generalized Euler scheme)
            u_ = u
            t = t + dt
        
        return u 
# This can be considered the transformed Python class from Fortran 
# to Python which is usually done through the Transformer class itself.

### Equinox as the Bridge to JAX

The key intermediary that bridges a regular Python class and a **JAX**-ready, **JIT**-compiled module is **Equinox**.

By using Equinox, we can transform standard Python object structures into JAX-compatible modules with only minimal modifications. This enables:

- Seamless integration with the JAX ecosystem.
- Efficient **Just-In-Time (JIT)** compilation.
- Automatic compatibility with JAX transformations such as differentiation, vectorization, and parallelization.
- Further program transformations while preserving a familiar object-oriented structure.

As a result, Equinox provides a lightweight and flexible pathway from conventional Python classes to high-performance, JAX-executable modules.

In [13]:
# Such transformation done resembles the code below. 
class Cost(eqx.Module):

    @eqx.filter_jit # <---- This automatically detects which parts of the class are static and which parts are JAX-traceable.
    def COST_FUNCTION(self, u):
        e = jnp.exp(1.0) # <---- Simple transformation from numpy to jax.numpy 
        j = (u - e) ** 2
        return j
    
    @eqx.filter_jit 
    def theta_step(self, u, theta, dt):
        return u * (1 + dt * (1 - theta)) / (1 - dt * theta)

    @eqx.filter_jit 
    def theta_method(self, theta):
        end_time = 1.0
        dt = 0.1
        u0 = 1.0

        N = int(end_time / dt)

        # Using lax.scan here because the number of timesteps is fixed, and scan is more efficient and XLA-optimizable than while_loop.
        def body(u, _):
            return self.theta_step(u, theta, dt), None

        u_final, _ = jax.lax.scan(body, u0, jnp.arange(N))
        return u_final
    
# Equinox makes it possible to treat class instances as PyTrees, meaning JAX can: trace the class, 
# differentiate through its fields, JIT-compile its methods, XLA JIT compilation (fast execution), 
# GPU/TPU compatibility

In [14]:
# The emulation process is performed by solving an ODE and optimizing its parameters using gradient descent.
cost_model = Cost()

@eqx.filter_jit
def cost_grad_fn(theta):
    u = cost_model.theta_method(theta)
    return cost_model.COST_FUNCTION(u)

grad_cost = jax.jit(jax.grad(cost_grad_fn))

def gradient_descent_jax(maxiter = 1000, gtol= 1e-05, alpha = 0.2, theta = 0.0): # Even the function for the ODE can be jitted to accelerate even further 
    jd_ = 1.0
    for i in range(maxiter):
        if i == maxiter:
            print(f'Reached maximum iterations without convergence')
            break
        
        jd = grad_cost(theta)
        theta = theta - alpha * jd
        
        if np.abs(jd - jd_) < gtol:
            print(f'Converged in {i} iterations due to gradient convergence')
            break

        jd_ = jd

In [15]:
gradient_descent_jax()

Converged in 188 iterations due to gradient convergence


In [14]:
%timeit -n 1 -r 1 gradient_descent_jax()

Converged in 188 iterations due to gradient convergence
8.66 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
